<a href="https://colab.research.google.com/github/busracil/son/blob/claude%2Fman-turkey-stock-prediction-G6cqy/man_stock_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚛 MAN TÜRKİYE STOK TAHMİN PROJESİ
**Model:** Parça bazlı en iyi model seçimi (ML + Croston varyantları)
**Hedef:** WAPE < %25

In [1]:
# Hücre 1: Kütüphane Yükleme
!pip install xgboost lightgbm catboost tqdm plotly streamlit -q

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print('✅ Tüm kütüphaneler yüklendi!')
print(f'XGBoost: {xgb.__version__}')
print(f'LightGBM: {lgb.__version__}')
print(f'CatBoost: {cb.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.4 MB/s eta 0:00:00
✅ Tüm kütüphaneler yüklendi!
XGBoost: 3.2.0
LightGBM: 4.6.0
CatBoost: 1.2.10


In [ ]:
# Hücre 2: Excel Yükleme ve Veri Analizi
from google.colab import files

# Dosyayı yükle
print('📂 Excel dosyasını yükleyin (MAN_ML_Dataset_v4.xlsx)...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename, engine='openpyxl')

# Tarih dönüşümü
df['Tarih'] = pd.to_datetime(df['Tarih'])

print(f'\n📊 VERİ YAPISI')
print(f'{'='*50}')
print(f'Toplam satır       : {len(df):,}')
print(f'Toplam sütun       : {df.shape[1]}')
print(f'Benzersiz parça    : {df["Parça_Kodu"].nunique():,}')
print(f'Tarih aralığı      : {df["Tarih"].min().strftime("%Y-%m")} / {df["Tarih"].max().strftime("%Y-%m")}')
print(f'Sıfır talep oranı  : %{(df["Talep"] == 0).mean()*100:.1f}')
print(f'Aralıklı talep     : %{df["is_intermittent"].mean()*100:.1f}')

print(f'\n📋 SÜTUNLAR: {list(df.columns)}')
print(f'\n📈 TALEP İSTATİSTİKLERİ:')
print(df['Talep'].describe())

print(f'\n🏷️ SEGMENT DAĞILIMI:')
if 'Segment' in df.columns:
    print(df.groupby('Segment')['Parça_Kodu'].nunique())

print(f'\n🔤 ABC DAĞILIMI:')
if 'ABC' in df.columns:
    print(df.groupby('ABC')['Parça_Kodu'].nunique())

print(f'\n📊 SPLIT DAĞILIMI:')
print(df['Split'].value_counts())

📂 Excel dosyasını yükleyin (MAN_ML_Dataset_v4.xlsx)...


In [ ]:
# Hücre 3: Metrik Fonksiyonları

def calculate_wape(y_true, y_pred):
    """Weighted Absolute Percentage Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return (np.sum(np.abs(y_true - y_pred)) / denom) * 100

def calculate_rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def calculate_mae(y_true, y_pred):
    """Mean Absolute Error"""
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def calculate_bias(y_true, y_pred):
    """Forecast Bias"""
    return np.mean(np.array(y_pred) - np.array(y_true))

def calculate_all_metrics(y_true, y_pred):
    """Tüm metrikleri hesapla"""
    return {
        'WAPE': calculate_wape(y_true, y_pred),
        'RMSE': calculate_rmse(y_true, y_pred),
        'MAE': calculate_mae(y_true, y_pred),
        'Bias': calculate_bias(y_true, y_pred)
    }

print('✅ Metrik fonksiyonları tanımlandı!')

# Test
y_test = np.array([10, 20, 30, 0, 15])
y_pred = np.array([12, 18, 28, 2, 14])
metrics = calculate_all_metrics(y_test, y_pred)
print(f'\n🧪 Test metrikleri: {metrics}')

In [ ]:
# Hücre 4: Train/Test Split

train_df = df[df['Split'] == 'Train'].copy()
test_df  = df[df['Split'] == 'Test'].copy()

print(f'📊 SPLIT SONUÇLARI')
print(f'{'='*50}')
print(f'Train seti         : {len(train_df):,} satır')
print(f'Test seti          : {len(test_df):,} satır')
print(f'Train tarih aralığı: {train_df["Tarih"].min().strftime("%Y-%m")} - {train_df["Tarih"].max().strftime("%Y-%m")}')
print(f'Test tarih aralığı : {test_df["Tarih"].min().strftime("%Y-%m")} - {test_df["Tarih"].max().strftime("%Y-%m")}')

unique_parts = df['Parça_Kodu'].unique()
intermittent_parts = df[df['is_intermittent'] == 1]['Parça_Kodu'].unique()
regular_parts = df[df['is_intermittent'] == 0]['Parça_Kodu'].unique()

print(f'\n🔢 PARÇA DAĞILIMI')
print(f'Toplam parça       : {len(unique_parts):,}')
print(f'Aralıklı parça     : {len(intermittent_parts):,} (%{len(intermittent_parts)/len(unique_parts)*100:.1f})')
print(f'Normal parça       : {len(regular_parts):,} (%{len(regular_parts)/len(unique_parts)*100:.1f})')

# Feature sütunlarını belirle
EXCLUDE_COLS = ['Parça_Kodu', 'Tarih', 'Split', 'Talep', 'Talep_log',
                'is_intermittent', 'ABC', 'XYZ', 'Segment']

FEATURES = [col for col in df.columns
            if col not in EXCLUDE_COLS
            and df[col].dtype in ['float64', 'int64', 'float32', 'int32']]

print(f'\n⚙️ FEATURE SÜTUNLARI ({len(FEATURES)} adet):')
print(FEATURES)

In [ ]:
# Hücre 5: Model Eğitim Fonksiyonları

# ─── ML Modelleri ────────────────────────────────────────────

def train_xgboost(X_train, y_train_log, sample_weights):
    model = xgb.XGBRegressor(
        max_depth=5, learning_rate=0.05, n_estimators=150,
        subsample=0.8, colsample_bytree=0.8, verbosity=0,
        random_state=42
    )
    model.fit(X_train, y_train_log, sample_weight=sample_weights)
    return model

def train_lightgbm(X_train, y_train_log, sample_weights):
    model = lgb.LGBMRegressor(
        max_depth=5, learning_rate=0.05, n_estimators=150,
        subsample=0.8, verbosity=-1, random_state=42
    )
    model.fit(X_train, y_train_log, sample_weight=sample_weights)
    return model

def train_catboost(X_train, y_train_log, sample_weights):
    model = cb.CatBoostRegressor(
        depth=5, learning_rate=0.05, iterations=150,
        verbose=False, random_seed=42
    )
    model.fit(X_train, y_train_log, sample_weight=sample_weights)
    return model

def train_randomforest(X_train, y_train_log, sample_weights):
    model = RandomForestRegressor(
        max_depth=8, n_estimators=100, random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train_log, sample_weight=sample_weights)
    return model


# ─── Croston Varyantları ──────────────────────────────────────

class CrostonModel:
    """Croston ve varyantları (SBA, TSB) için temel sınıf"""

    def __init__(self, alpha=0.1, method='croston'):
        self.alpha = alpha
        self.method = method  # 'croston', 'sba', 'tsb'
        self.forecast_ = None

    def fit(self, demand):
        demand = np.array(demand, dtype=float)
        n = len(demand)
        alpha = self.alpha

        # İlk sıfır olmayan değeri bul
        nz_idx = np.where(demand > 0)[0]
        if len(nz_idx) == 0:
            self.forecast_ = 0.0
            return self

        # Başlangıç değerleri
        z = demand[nz_idx[0]]   # demand level
        p = nz_idx[0] + 1 if nz_idx[0] > 0 else 1  # interval
        q = 1 / p               # TSB için occurrence probability

        interval = p
        for i in range(1, n):
            if demand[i] > 0:
                z = alpha * demand[i] + (1 - alpha) * z
                p = alpha * interval + (1 - alpha) * p
                q = alpha * 1 + (1 - alpha) * q
                interval = 1
            else:
                q = alpha * 0 + (1 - alpha) * q
                interval += 1

        if self.method == 'croston':
            self.forecast_ = z / p
        elif self.method == 'sba':
            self.forecast_ = (1 - alpha / 2) * z / p
        elif self.method == 'tsb':
            self.forecast_ = q * z
        else:
            self.forecast_ = z / p

        return self

    def predict(self, n_periods):
        return np.full(n_periods, max(self.forecast_, 0))


def train_croston(demand_history):
    model = CrostonModel(alpha=0.1, method='croston')
    model.fit(demand_history)
    return model

def train_sba(demand_history):
    model = CrostonModel(alpha=0.1, method='sba')
    model.fit(demand_history)
    return model

def train_tsb(demand_history):
    model = CrostonModel(alpha=0.1, method='tsb')
    model.fit(demand_history)
    return model


print('✅ Tüm model fonksiyonları tanımlandı!')
print('  ML modelleri  : XGBoost, LightGBM, CatBoost, RandomForest')
print('  Croston grubu : Croston, SBA, TSB')

In [ ]:
# Hücre 6: Model Karşılaştırma ve Seçim Fonksiyonu

def select_best_model_for_part(part_code, train_df, test_df, is_intermittent):
    """
    Bir parça için tüm uygun modelleri eğit, test et, en iyisini seç.

    Returns:
        best_model_name : str
        best_predictions: array
        best_wape       : float
        all_results     : dict {model_adı: wape}
    """
    part_train = train_df[train_df['Parça_Kodu'] == part_code].sort_values('Tarih')
    part_test  = test_df[test_df['Parça_Kodu'] == part_code].sort_values('Tarih')

    if len(part_test) == 0:
        return None, np.array([]), np.nan, {}

    y_test_true = part_test['Talep'].values
    results = {}
    predictions_dict = {}

    if is_intermittent:
        # ── Croston varyantları ──
        model_funcs = {
            'Croston': train_croston,
            'SBA'    : train_sba,
            'TSB'    : train_tsb
        }
        demand_history = part_train['Talep'].values

        for model_name, train_func in model_funcs.items():
            try:
                model = train_func(demand_history)
                pred  = model.predict(len(part_test))
                wape  = calculate_wape(y_test_true, pred)
                results[model_name] = wape if not np.isnan(wape) else 9999
                predictions_dict[model_name] = pred
            except Exception:
                results[model_name] = 9999
                predictions_dict[model_name] = np.zeros(len(part_test))

    else:
        # ── ML modelleri ──
        X_train = part_train[FEATURES].fillna(0)
        y_train_log = part_train['Talep_log'].values
        X_test  = part_test[FEATURES].fillna(0)

        # Sample weights
        weight_map = {2022: 1.0, 2023: 2.0, 2024: 5.0}
        weights = part_train['Yil'].map(weight_map).fillna(1.0).values

        if len(X_train) < 3:
            # Veri yetersizse sıfır tahmin
            for mn in ['XGBoost', 'LightGBM', 'CatBoost', 'RandomForest']:
                results[mn] = 9999
                predictions_dict[mn] = np.zeros(len(part_test))
        else:
            model_funcs = {
                'XGBoost'     : train_xgboost,
                'LightGBM'    : train_lightgbm,
                'CatBoost'    : train_catboost,
                'RandomForest': train_randomforest
            }

            for model_name, train_func in model_funcs.items():
                try:
                    model    = train_func(X_train, y_train_log, weights)
                    pred_log = model.predict(X_test)
                    pred     = np.expm1(pred_log)
                    pred     = np.maximum(pred, 0)
                    wape     = calculate_wape(y_test_true, pred)
                    results[model_name] = wape if not np.isnan(wape) else 9999
                    predictions_dict[model_name] = pred
                except Exception:
                    results[model_name] = 9999
                    predictions_dict[model_name] = np.zeros(len(part_test))

    # En iyi modeli seç (en düşük WAPE)
    best_model_name = min(results, key=results.get)
    best_wape       = results[best_model_name]
    best_predictions = predictions_dict[best_model_name]

    # 9999 sonsuz olarak işaretle
    clean_results = {k: (v if v != 9999 else np.nan) for k, v in results.items()}

    return best_model_name, best_predictions, (best_wape if best_wape != 9999 else np.nan), clean_results


print('✅ select_best_model_for_part() fonksiyonu tanımlandı!')

In [ ]:
# Hücre 7: Tüm Parçalar İçin Model Eğitimi ve Seçimi

all_predictions    = []
model_selection_log = []

# Her parça için is_intermittent değerini hızlıca al
part_intermittent = (
    df.groupby('Parça_Kodu')['is_intermittent']
    .first()
    .to_dict()
)

errors = []

for part_code in tqdm(unique_parts, desc='Parçalar işleniyor'):
    try:
        is_intermittent = int(part_intermittent.get(part_code, 0))

        best_model, predictions, best_wape, all_wapes = select_best_model_for_part(
            part_code, train_df, test_df, is_intermittent
        )

        if best_model is None:
            continue

        # Sonuç DataFrame
        part_test_sorted = test_df[test_df['Parça_Kodu'] == part_code].sort_values('Tarih')

        keep_cols = ['Parça_Kodu', 'Tarih', 'Talep']
        for c in ['ABC', 'XYZ', 'Segment']:
            if c in part_test_sorted.columns:
                keep_cols.append(c)

        result_df = part_test_sorted[keep_cols].copy()
        result_df['Tahmin']         = predictions
        result_df['Selected_Model'] = best_model
        result_df['Model_WAPE']     = best_wape
        result_df['is_intermittent'] = is_intermittent

        all_predictions.append(result_df)

        # Model seçim logu
        log_entry = {
            'Parça_Kodu'      : part_code,
            'is_intermittent' : is_intermittent,
            'Selected_Model'  : best_model,
            'WAPE'            : best_wape,
            'XGBoost_WAPE'    : all_wapes.get('XGBoost',     np.nan),
            'LightGBM_WAPE'   : all_wapes.get('LightGBM',    np.nan),
            'CatBoost_WAPE'   : all_wapes.get('CatBoost',    np.nan),
            'RandomForest_WAPE': all_wapes.get('RandomForest', np.nan),
            'Croston_WAPE'    : all_wapes.get('Croston',     np.nan),
            'SBA_WAPE'        : all_wapes.get('SBA',         np.nan),
            'TSB_WAPE'        : all_wapes.get('TSB',         np.nan),
        }
        model_selection_log.append(log_entry)

    except Exception as e:
        errors.append({'part': part_code, 'error': str(e)})

final_predictions  = pd.concat(all_predictions, ignore_index=True)
model_selection_df = pd.DataFrame(model_selection_log)

print(f'\n✅ MODEL EĞİTİMİ TAMAMLANDI!')
print(f'İşlenen parça      : {len(model_selection_log):,}')
print(f'Hata               : {len(errors)}')
print(f'Tahmin satırı      : {len(final_predictions):,}')

In [ ]:
# Hücre 8: Genel Değerlendirme ve Model İstatistikleri

print('='*70)
print('                MODEL KULLANIM İSTATİSTİKLERİ')
print('        (En iyi model olarak seçilme sayısı ve oranı)')
print('='*70)

model_counts = model_selection_df['Selected_Model'].value_counts()
total_parts  = len(model_selection_df)

# Her model için seçilme sayısı ve ort. WAPE
model_stats = (
    model_selection_df
    .groupby('Selected_Model')
    .agg(
        Seçilme=('Selected_Model', 'count'),
        Ort_WAPE=('WAPE', 'mean'),
        Min_WAPE=('WAPE', 'min'),
        Max_WAPE=('WAPE', 'max')
    )
    .reset_index()
    .sort_values('Seçilme', ascending=False)
)
model_stats['Oran'] = model_stats['Seçilme'] / total_parts * 100

for _, row in model_stats.iterrows():
    star = ' ⭐ EN ÇOK SEÇİLEN' if row['Seçilme'] == model_stats['Seçilme'].max() else ''
    print(f"{row['Selected_Model']:<15}: {row['Seçilme']:>5} parça "
          f"(%{row['Oran']:.1f}) - Ort. WAPE: %{row['Ort_WAPE']:.1f}{star}")

print()

# Genel performans
overall_wape = calculate_wape(final_predictions['Talep'], final_predictions['Tahmin'])
overall_rmse = calculate_rmse(final_predictions['Talep'], final_predictions['Tahmin'])
overall_mae  = calculate_mae(final_predictions['Talep'], final_predictions['Tahmin'])

print('='*70)
print('          GENEL PERFORMANS (En iyi modeller kullanılarak)')
print('='*70)
target_ok = '✅' if overall_wape < 25 else '❌'
print(f'WAPE : %{overall_wape:.2f}  {target_ok} (Hedef: <%25)')
print(f'RMSE : {overall_rmse:.2f}')
print(f'MAE  : {overall_mae:.2f}')

print()

# Segment bazlı performans
if 'Segment' in final_predictions.columns:
    print('='*70)
    print('                    SEGMENT BAZLI PERFORMANS')
    print('='*70)
    for seg, grp in final_predictions.groupby('Segment'):
        seg_wape = calculate_wape(grp['Talep'], grp['Tahmin'])
        print(f'  {str(seg):<12}: WAPE=%{seg_wape:.1f}  (n={len(grp):,})')

# ── Görselleştirme ──────────────────────────────────────────
fig1 = px.pie(
    model_stats, values='Seçilme', names='Selected_Model',
    title='Model Kullanım Dağılımı (En İyi Model Seçilme Sayısı)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig1.show()

fig2 = px.bar(
    model_stats.sort_values('Ort_WAPE'),
    x='Selected_Model', y='Ort_WAPE',
    title='Model Bazlı Ortalama WAPE (Seçilen Parçalarda)',
    labels={'Ort_WAPE': 'Ortalama WAPE (%)'},
    color='Ort_WAPE',
    color_continuous_scale='RdYlGn_r',
    text_auto='.1f'
)
fig2.add_hline(y=25, line_dash='dash', line_color='red',
               annotation_text='Hedef: %25')
fig2.show()

In [ ]:
# Hücre 9: Stok Optimizasyonu - (s, Q) Hesaplama
# EOQ tabanlı (s,Q) politikası

inventory_params = []

for part_code in tqdm(unique_parts, desc='Stok parametreleri hesaplanıyor'):
    part_pred = final_predictions[final_predictions['Parça_Kodu'] == part_code]
    part_all  = df[df['Parça_Kodu'] == part_code]

    if len(part_pred) == 0:
        continue

    # Maliyet parametreleri
    lead_time  = part_all['Tedarik Süresi (gün)'].iloc[0]  if 'Tedarik Süresi (gün)'   in part_all.columns else 30
    unit_cost  = part_all['Birim Maliyet (TL)'].iloc[0]    if 'Birim Maliyet (TL)'      in part_all.columns else 1000
    order_cost = part_all['Sipariş Maliyeti (TL)'].iloc[0] if 'Sipariş Maliyeti (TL)'   in part_all.columns else 500
    hold_cost  = part_all['Elde Tutma (TL/adet/ay)'].iloc[0] if 'Elde Tutma (TL/adet/ay)' in part_all.columns else 50

    lead_time  = max(float(lead_time),  1)
    unit_cost  = max(float(unit_cost),  1)
    order_cost = max(float(order_cost), 1)
    hold_cost  = max(float(hold_cost),  0.01)

    # Ortalama aylık talep (tahminlerden)
    mean_demand = part_pred['Tahmin'].mean()
    std_demand  = part_pred['Tahmin'].std() if len(part_pred) > 1 else 0

    if mean_demand <= 0:
        eoq = 0
        rop = 0
        safety_stock = 0
    else:
        # EOQ hesabı (aylık talep → yıllık)
        annual_demand = mean_demand * 12
        annual_hold   = hold_cost * 12
        eoq = np.sqrt((2 * annual_demand * order_cost) / annual_hold)

        # Lead time talebi (gün → ay dönüşümü)
        lead_time_months = lead_time / 30
        lead_demand = mean_demand * lead_time_months

        # %95 servis seviyesi → z=1.645
        safety_stock = 1.645 * std_demand * np.sqrt(lead_time_months)
        rop = lead_demand + safety_stock

    row = part_all[['ABC', 'XYZ', 'Segment']].iloc[0].to_dict() if {'ABC','XYZ','Segment'}.issubset(part_all.columns) else {}
    row.update({
        'Parça_Kodu'       : part_code,
        'Ort_Talep_Ay'     : round(mean_demand, 2),
        'Std_Talep_Ay'     : round(std_demand,  2),
        'EOQ'              : round(eoq, 1),
        'Güvenlik_Stoku'   : round(safety_stock, 1),
        'Yeniden_Sipariş_N': round(rop, 1),
        'Lead_Time_Gun'    : lead_time,
        'Birim_Maliyet_TL' : unit_cost,
        'Siparis_Maliyeti' : order_cost,
        'Elde_Tutma_Maliyet': hold_cost,
    })
    inventory_params.append(row)

inventory_df = pd.DataFrame(inventory_params)

print('\n✅ STOK OPTİMİZASYONU TAMAMLANDI!')
print(inventory_df.describe())

In [ ]:
# Hücre 10: Dosyaları Kaydet

final_predictions.to_csv('final_predictions.csv', index=False)
model_selection_df.to_csv('model_selection_summary.csv', index=False)
inventory_df.to_csv('inventory_parameters.csv', index=False)
model_stats.to_csv('model_stats_summary.csv', index=False)

print('✅ DOSYALAR KAYDEDİLDİ:')
print('  📄 final_predictions.csv      - Tahmin sonuçları')
print('  📄 model_selection_summary.csv - Model seçim özeti')
print('  📄 inventory_parameters.csv   - Stok parametreleri')
print('  📄 model_stats_summary.csv    - Model istatistikleri')

# Dosyaları indir
from google.colab import files
files.download('final_predictions.csv')
files.download('model_selection_summary.csv')
files.download('inventory_parameters.csv')
files.download('model_stats_summary.csv')

In [ ]:
# Hücre 11: GitHub Push

GITHUB_TOKEN = 'YOUR_GITHUB_TOKEN'  # @param {type:"string"}
GITHUB_USER  = 'busracil'            # @param {type:"string"}
REPO_NAME    = 'son'                 # @param {type:"string"}
BRANCH       = 'claude/man-turkey-stock-prediction-G6cqy'  # @param {type:"string"}

import subprocess, shutil, os

repo_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'

cmds = [
    f'git clone --branch {BRANCH} {repo_url} /tmp/repo',
    'cp final_predictions.csv /tmp/repo/',
    'cp model_selection_summary.csv /tmp/repo/',
    'cp inventory_parameters.csv /tmp/repo/',
    'cp model_stats_summary.csv /tmp/repo/',
    'cd /tmp/repo && git config user.email "colab@man-turkey.com"',
    'cd /tmp/repo && git config user.name "MAN Turkey Colab"',
    'cd /tmp/repo && git add .',
    'cd /tmp/repo && git commit -m "feat: add ML predictions and inventory parameters"',
    f'cd /tmp/repo && git push origin {BRANCH}',
]

for cmd in cmds:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'⚠️  {cmd[:50]}...\n   HATA: {result.stderr[:200]}')
    else:
        print(f'✅ {cmd[:60]}')

print('\n🚀 GitHub push tamamlandı!')

In [ ]:
# Hücre 12: Streamlit + LocalTunnel ile Dashboard Başlatma

!pip install streamlit localtunnel -q

# app.py dosyasını GitHub'dan al
!wget -q https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/app.py -O app.py

# CSV'lerin mevcut olduğunu doğrula
import os
for f in ['final_predictions.csv', 'model_selection_summary.csv', 'inventory_parameters.csv']:
    print(f'{'✅' if os.path.exists(f) else '❌'} {f}')

# Streamlit'i arka planda başlat
import subprocess, threading, time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'app.py',
                    '--server.port', '8501',
                    '--server.headless', 'true'], check=False)

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(5)

# LocalTunnel ile public URL oluştur
!npx localtunnel --port 8501 &
time.sleep(5)
print('\n🌐 Dashboard URL yukarıda görünüyor!')
print('   (lt.xxx.loca.lt bağlantısını kopyalayın)')